# Markov Decision Processes

**Companion lesson:** https://ml-viz.vercel.app/courses/reinforcement-learning/01-markov-decision-processes

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A gridworld MDP

We define the environment, then solve it with **value iteration** — repeatedly applying the Bellman optimality update until the value function stops changing.

In [ ]:
class GridWorld:
    """n x n grid. Start top-left (0), goal bottom-right. Actions: 0=up 1=down 2=left 3=right.
    Reward -1 per step, +10 at the goal (terminal)."""
    def __init__(self, n=5):
        self.n = n; self.nS = n*n; self.nA = 4; self.goal = n*n-1
    def step(self, s, a):
        r, c = divmod(s, self.n)
        if a==0: r = max(0, r-1)
        elif a==1: r = min(self.n-1, r+1)
        elif a==2: c = max(0, c-1)
        else: c = min(self.n-1, c+1)
        s2 = r*self.n + c
        done = (s2 == self.goal)
        return s2, (10.0 if done else -1.0), done

env = GridWorld(5)
print('states:', env.nS, '| actions:', env.nA, '| goal:', env.goal)

## Value iteration

$V(s)\leftarrow\max_a\big[r(s,a)+\gamma V(s')\big]$ for the (deterministic) gridworld, iterated to convergence.

In [ ]:
def value_iteration(env, gamma=0.9, tol=1e-6):
    V = np.zeros(env.nS)
    for it in range(1000):
        V_new = V.copy()
        for s in range(env.nS):
            if s == env.goal: continue
            V_new[s] = max(env.step(s,a)[1] + gamma*V[env.step(s,a)[0]] for a in range(env.nA))
        if np.max(np.abs(V_new - V)) < tol:
            print(f'converged in {it} iterations'); V = V_new; break
        V = V_new
    return V

V = value_iteration(env)
print('V* grid:'); print(np.round(V.reshape(5,5), 1))

## Extract and visualize the optimal policy

The greedy policy w.r.t. $V^*$ — the best action in every cell.

In [ ]:
def greedy_policy(env, V, gamma=0.9):
    pi = np.zeros(env.nS, int)
    for s in range(env.nS):
        pi[s] = np.argmax([env.step(s,a)[1] + gamma*V[env.step(s,a)[0]] for a in range(env.nA)])
    return pi

arrows = {0:'↑',1:'↓',2:'←',3:'→'}
pi = greedy_policy(env, V)
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(V.reshape(5,5), cmap='viridis')
for s in range(env.nS):
    r,c = divmod(s, env.n)
    ax.text(c, r, 'G' if s==env.goal else arrows[pi[s]], ha='center', va='center', color='w', fontsize=16)
ax.set_title('Optimal value (color) + policy (arrows)'); ax.axis('off'); plt.show()

## Discounting shapes the values

In [ ]:
for g in [0.5, 0.9, 0.99]:
    Vg = value_iteration(env, gamma=g)
    print(f'gamma={g}: V(start)={Vg[0]:.2f}')

## Key takeaways

- An MDP is states, actions, transitions, rewards, and a discount $\gamma$.
- **Value iteration** applies the Bellman optimality update until $V$ converges.
- The optimal policy is **greedy** with respect to $V^*$.
- $\gamma$ sets the horizon: larger $\gamma$ values distant rewards more.

## ✏️ Your turn

### Exercise 1 — One Bellman backup step

The Bellman optimality equation for value iteration:

$$V(s) \\leftarrow \\max_a \\left[r(s,a) + \\gamma \\, V(s')\\right]$$

For a deterministic transition $s \\xrightarrow{a} s'$ this is a scalar operation.
Implement it and verify on hand-checkable fixtures.

In [ ]:
def bellman_backup(r, gamma, v_next):
    """Single Bellman backup: return r + gamma * v_next."""
    # TODO(you): one line
    ...

In [ ]:
assert abs(bellman_backup(-1, 0.9, 0.0) - (-1.0)) < 1e-9, \
    "single step to terminal state (V=0): backup = -1"
assert abs(bellman_backup(10, 0.9, 0.0) - 10.0) < 1e-9, \
    "goal transition with zero-value terminal: backup = reward"
assert abs(bellman_backup(-1, 0.9, 5.0) - 3.5) < 1e-9, \
    "bootstrap from non-zero V: -1 + 0.9*5 = 3.5"
assert abs(bellman_backup(0, 1.0, 3.0) - 3.0) < 1e-9, \
    "zero reward + gamma=1: backup equals next value"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bellman_backup(r, gamma, v_next):
    return r + gamma * v_next
```

</details>

### Exercise 2 — Discount factor shapes the effective horizon

The effective horizon is approximately $\\frac{1}{1-\\gamma}$: with $\\gamma=0.9$ an agent looks
about 10 steps ahead; with $\\gamma=0.99$, about 100 steps. Verify this by computing the
discounted sum of a constant reward stream.

In [ ]:
def discounted_sum(r, gamma, n_steps):
    """Sum of discounted constant reward: r * sum(gamma^t for t in 0..n_steps-1).
    Returns the geometric-series value."""
    # TODO(you): implement (closed-form or loop, your choice)
    ...

In [ ]:
import math

# With gamma=0, only the immediate reward matters
assert abs(discounted_sum(1.0, 0.0, 100) - 1.0) < 1e-9, \
    "gamma=0 means only immediate reward counts"
# Closed-form: r * (1 - gamma^n) / (1 - gamma)
for gamma in (0.5, 0.9, 0.99):
    n = 1000
    expected = 1.0 * (1 - gamma**n) / (1 - gamma)
    assert abs(discounted_sum(1.0, gamma, n) - expected) < 1e-4, \
        f"discounted sum for gamma={gamma} should equal geometric series"
# Higher gamma means larger sum (more future rewards counted)
assert discounted_sum(1.0, 0.5, 50) < discounted_sum(1.0, 0.9, 50) < discounted_sum(1.0, 0.99, 50), \
    "higher gamma gives larger discounted sum"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def discounted_sum(r, gamma, n_steps):
    if gamma == 0.0:
        return r
    return r * (1 - gamma**n_steps) / (1 - gamma)
```

</details>